# ASL Sign Detection - PyTorch Lightning

## Leave an upvote if you find it useful or fork it :) And also for the notebooks mentioned in the Sources section

## Versions:
- 11: Fix `reset_index` bug when setting up `train_df` and `val_df` in `ASLDataModule` in combination with `__getitem()__`, which caused unrealistic high val accuracies:
```python
    def setup(self, stage=None):
        val_fold = self.hparams.val_fold
        
        train_df = self.df[self.df.fold != val_fold].reset_index(drop=True) # Removed .reset_index in version 11
        val_df = self.df[self.df.fold == val_fold].reset_index(drop=True) # Removed .reset_index in version 11
```

```python
    def __getitem__(self, idx):
        # Use df_index as idx due to folds splitting
        df_index = self.df.index.values[idx]

        x = self.X[df_index]
        y = self.y[df_index]
```       
- 14: Use lip instead of face features
- 15: 
    - Use [GISLR Feature Data: On the Shoulders](https://www.kaggle.com/code/roberthatch/gislr-feature-data-on-the-shoulders) features
    - Use TF Preprocessing from [GISLR Feature Data: On the Shoulders](https://www.kaggle.com/code/roberthatch/gislr-feature-data-on-the-shoulders)

## TODO: 
- [ ] Implement FeatureGen in PyTorch
- [ ] Use ArcMarginProduct
- [ ] Use 2D instead of 3D points
- [x] Switch to using time-based features instead of only mean and std
- [ ] Add mirror data augmentation

## Sources:
- ### [Mayukh Bhattacharyya](https://www.kaggle.com/mayukh18)'s [End-to-End Pytorch Training + Submission](https://www.kaggle.com/code/mayukh18/end-to-end-pytorch-training-submission)
    - Great feature generation using PyTorch model
    - Super nice PyTorch to TFLite Conversion
- ### [Darien Schettler](https://www.kaggle.com/dschettler8845)'s [🤟 GISLR 🤟 - 📚Learn – 🔭EDA – 🤖Baseline](https://www.kaggle.com/code/dschettler8845/gislr-learn-eda-baseline)
    - Awesome EDA to understand training data
    - Nice and simple baseline model
- ### [GISLR Feature Data: On the Shoulders](https://www.kaggle.com/code/roberthatch/gislr-feature-data-on-the-shoulders)
    - Feature creation
    - TF Preprocessing

# Installs

In [ ]:
!pip install onnx-tf tflite-runtime nb_black -q
!pip install --upgrade -q wandb

In [ ]:
%load_ext lab_black

# Imports

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import onnx
import pandas as pd
import pytorch_lightning as pl
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import wandb
from kaggle_secrets import UserSecretsClient
from onnx_tf.backend import prepare
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.loggers import WandbLogger
from sklearn.model_selection import StratifiedGroupKFold
from timm.optim import create_optimizer_v2
from torchmetrics import MetricCollection

# WandB

In [ ]:
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("wandb_api")
wandb.login(key=wandb_api)

# Paths & Settings

In [ ]:
KAGGLE_DIR = Path("/") / "kaggle"
INPUT_DIR = KAGGLE_DIR / "input"
DATA_ROOT_DIR = INPUT_DIR / "asl-signs"
NPY_DATA_DIR = INPUT_DIR / "gislr-feature-data-on-the-shoulders"
MY_NPY_DATA_DIR = INPUT_DIR / "asl-signs-features-npy"

TRAIN_CSV_PATH = MY_NPY_DATA_DIR / "train_prepared.csv"
SIGN_TO_IDX_PATH = INPUT_DIR / "asl-signs" / "sign_to_prediction_index_map.json"

with open(SIGN_TO_IDX_PATH, "r") as f:
    SIGN_TO_IDX = json.load(f)

N_SPLITS = 5
SEED = 2023
ROWS_PER_FRAME = 543

IN_FEATURES = 5796

In [ ]:
df = pd.read_csv(TRAIN_CSV_PATH)
df

# TF Feature Generator

## Source: [GISLR Feature Data: On the Shoulders](https://www.kaggle.com/code/roberthatch/gislr-feature-data-on-the-shoulders)

In [ ]:
DROP_Z = False

NUM_FRAMES = 15
SEGMENTS = 3

LEFT_HAND_OFFSET = 468
POSE_OFFSET = LEFT_HAND_OFFSET + 21
RIGHT_HAND_OFFSET = POSE_OFFSET + 33

## average over the entire face, and the entire 'pose'
averaging_sets = [[0, 468], [POSE_OFFSET, 33]]

lip_landmarks = [
    61,
    185,
    40,
    39,
    37,
    0,
    267,
    269,
    270,
    409,
    291,
    146,
    91,
    181,
    84,
    17,
    314,
    405,
    321,
    375,
    78,
    191,
    80,
    81,
    82,
    13,
    312,
    311,
    310,
    415,
    95,
    88,
    178,
    87,
    14,
    317,
    402,
    318,
    324,
    308,
]
left_hand_landmarks = list(range(LEFT_HAND_OFFSET, LEFT_HAND_OFFSET + 21))
right_hand_landmarks = list(range(RIGHT_HAND_OFFSET, RIGHT_HAND_OFFSET + 21))

point_landmarks = [
    item
    for sublist in [lip_landmarks, left_hand_landmarks, right_hand_landmarks]
    for item in sublist
]

LANDMARKS = len(point_landmarks) + len(averaging_sets)
print(LANDMARKS)
if DROP_Z:
    INPUT_SHAPE = (NUM_FRAMES, LANDMARKS * 2)
else:
    INPUT_SHAPE = (NUM_FRAMES, LANDMARKS * 3)

FLAT_INPUT_SHAPE = (INPUT_SHAPE[0] + 2 * (SEGMENTS + 1)) * INPUT_SHAPE[1]

In [ ]:
def tf_nan_mean(x, axis=0):
    return tf.reduce_sum(
        tf.where(tf.math.is_nan(x), tf.zeros_like(x), x), axis=axis
    ) / tf.reduce_sum(
        tf.where(tf.math.is_nan(x), tf.zeros_like(x), tf.ones_like(x)), axis=axis
    )


def tf_nan_std(x, axis=0):
    d = x - tf_nan_mean(x, axis=axis)
    return tf.math.sqrt(tf_nan_mean(d * d, axis=axis))


def flatten_means_and_stds(x, axis=0):
    # Get means and stds
    x_mean = tf_nan_mean(x, axis=0)
    x_std = tf_nan_std(x, axis=0)

    x_out = tf.concat([x_mean, x_std], axis=0)
    x_out = tf.reshape(x_out, (1, INPUT_SHAPE[1] * 2))
    x_out = tf.where(tf.math.is_finite(x_out), x_out, tf.zeros_like(x_out))
    return x_out

In [ ]:
class FeatureGenTF(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def call(self, x_in):
        if DROP_Z:
            x_in = x_in[:, :, 0:2]
        x_list = [
            tf.expand_dims(
                tf_nan_mean(x_in[:, av_set[0] : av_set[0] + av_set[1], :], axis=1),
                axis=1,
            )
            for av_set in averaging_sets
        ]
        x_list.append(tf.gather(x_in, point_landmarks, axis=1))
        x = tf.concat(x_list, 1)

        x_padded = x
        for i in range(SEGMENTS):
            p0 = tf.where(
                ((tf.shape(x_padded)[0] % SEGMENTS) > 0) & ((i % 2) != 0), 1, 0
            )
            p1 = tf.where(
                ((tf.shape(x_padded)[0] % SEGMENTS) > 0) & ((i % 2) == 0), 1, 0
            )
            paddings = [[p0, p1], [0, 0], [0, 0]]
            x_padded = tf.pad(x_padded, paddings, mode="SYMMETRIC")
        x_list = tf.split(x_padded, SEGMENTS)
        x_list = [flatten_means_and_stds(_x, axis=0) for _x in x_list]

        x_list.append(flatten_means_and_stds(x, axis=0))

        ## Resize only dimension 0. Resize can't handle nan, so replace nan with that dimension's avg value to reduce impact.
        x = tf.image.resize(
            tf.where(tf.math.is_finite(x), x, tf_nan_mean(x, axis=0)),
            [NUM_FRAMES, LANDMARKS],
        )
        x = tf.reshape(x, (1, INPUT_SHAPE[0] * INPUT_SHAPE[1]))
        x = tf.where(tf.math.is_nan(x), tf.zeros_like(x), x)
        x_list.append(x)
        x = tf.concat(x_list, axis=1)
        return x

# Dataset

In [ ]:
class ASLDataFrameDataset(torch.utils.data.Dataset):
    def __init__(self, df, in_features, transform=None):
        self.df = df
        self.transform = transform

        print("Loading data...")
        self.X = np.load(NPY_DATA_DIR / "feature_data.npy")
        self.y = np.load(NPY_DATA_DIR / "feature_labels.npy")
        print(f"X shape: {self.X.shape}, y shape: {self.y.shape}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Use df_index as idx due to folds splitting
        df_index = self.df.index.values[idx]
        x = self.X[df_index]
        y = self.y[df_index]

        x = torch.Tensor(x)
        y = torch.Tensor([y]).long()

        if self.transform:
            x = self.transform(x)

        return x, y

# Lightning DataModule

In [ ]:
class ASLDataModule(pl.LightningDataModule):
    def __init__(
        self,
        batch_size: int,
        data_csv_path: str,
        in_features: int,
        num_workers: int,
        val_fold: float,
    ):
        super().__init__()

        self.save_hyperparameters()

        self.df = pd.read_csv(data_csv_path)

        self.train_transform, self.val_transform = self._init_transforms()

    def _init_transforms(self):
        train_transform = None
        val_transform = None

        return train_transform, val_transform

    def setup(self, stage=None):
        val_fold = self.hparams.val_fold
        train_df = self.df[self.df.fold != val_fold]
        val_df = self.df[self.df.fold == val_fold]

        if stage == "fit" or stage is None:
            self.train_dataset = self._dataset(train_df, self.train_transform)
            self.val_dataset = self._dataset(val_df, self.val_transform)

    def _dataset(self, df, transform):
        return ASLDataFrameDataset(df, self.hparams.in_features, transform=transform)

    def train_dataloader(self):
        return self._dataloader(self.train_dataset, train=True)

    def val_dataloader(self):
        return self._dataloader(self.val_dataset)

    def _dataloader(self, dataset, train=False):
        return torch.utils.data.DataLoader(
            dataset,
            batch_size=self.hparams.batch_size,
            shuffle=train,
            num_workers=self.hparams.num_workers,
            drop_last=train,
        )

# Arc Margin

In [ ]:
# From https://github.com/lyakaap/Landmark2019-1st-and-3rd-Place-Solution/blob/master/src/modeling/metric_learning.py
# Added type annotations, device, and 16bit support
class ArcMarginProduct(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        scale: float,
        margin: float,
        easy_margin: bool,
        ls_eps: float,
    ):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.scale = scale
        self.margin = margin
        self.ls_eps = ls_eps  # label smoothing
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.easy_margin = easy_margin
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(
        self, input: torch.Tensor, label: torch.Tensor, device: str = "cuda"
    ) -> torch.Tensor:
        # --------------------------- cos(theta) & phi(theta) ---------------------
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))

        # # Enable 16 bit precision
        # cosine = cosine.to(torch.float32)

        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        phi = cosine * self.cos_m - sine * self.sin_m
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # --------------------------- convert label to one-hot ---------------------
        # one_hot = torch.zeros(cosine.size(), requires_grad=True, device='cuda')
        one_hot = torch.zeros(cosine.size(), device=device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        if self.ls_eps > 0:
            one_hot = (1 - self.ls_eps) * one_hot + self.ls_eps / self.out_features
        # -------------torch.where(out_i = {x_i if condition_i else y_i) ------------
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.scale

        return output

# Model

## Source: [🤟 GISLR 🤟 - 📚Learn – 🔭EDA – 🤖Baseline](https://www.kaggle.com/code/dschettler8845/gislr-learn-eda-baseline)

In [ ]:
class ASLLinearModel(torch.nn.Module):
    def __init__(
        self,
        in_features: int,
        first_out_features: int,
        num_classes: int,
        num_blocks: int,
        drop_rate: float,
    ):
        super().__init__()

        blocks = []
        out_features = first_out_features
        for idx in range(num_blocks):
            if idx == num_blocks - 1:
                out_features = num_classes

            blocks.append(self._make_block(in_features, out_features, drop_rate))

            in_features = out_features
            out_features = out_features // 2

        self.model = nn.Sequential(*blocks)
        print(self.model)

    def _make_block(self, in_features, out_features, drop_rate):
        return nn.Sequential(
            nn.Linear(in_features, out_features),
            nn.BatchNorm1d(out_features),
            nn.ReLU(),
            nn.Dropout(drop_rate),
        )

    def forward(self, x):
        return self.model(x)

# Lightning Module

In [ ]:
class ASLModule(pl.LightningModule):
    def __init__(
        self,
        drop_rate: float,
        eta_min: float,
        first_out_features: int,
        learning_rate: float,
        loss: str,
        in_features: int,
        max_epochs: int,
        model_name: str,
        num_blocks: int,
        num_classes: int,
        optimizer: str,
        scheduler: str,
        weight_decay: float,
    ):
        super().__init__()

        self.save_hyperparameters()

        self.model = self._init_model()

        self.loss_fn = self._init_loss_fn()

        self.metrics = self._init_metrics()

    def _init_model(self):
        if self.hparams.model_name == "linear":
            return ASLLinearModel(
                in_features=self.hparams.in_features,
                first_out_features=self.hparams.first_out_features,
                num_classes=self.hparams.num_classes,
                num_blocks=self.hparams.num_blocks,
                drop_rate=self.hparams.drop_rate,
            )
        else:
            raise ValueError(f"{self.hparams.model_name} is not a valid model name")

    def _init_loss_fn(self):
        if self.hparams.loss == "CELoss":
            return torch.nn.CrossEntropyLoss()
        else:
            raise ValueError(f"{self.hparams.loss} is not a valid loss function")

    def _init_metrics(self):
        metrics = {
            "acc": torchmetrics.classification.MulticlassAccuracy(
                num_classes=len(SIGN_TO_IDX)
            ),
        }
        metric_collection = MetricCollection(metrics)

        return torch.nn.ModuleDict(
            {
                "train_metrics": metric_collection.clone(prefix="train_"),
                "val_metrics": metric_collection.clone(prefix="val_"),
            }
        )

    def configure_optimizers(self):
        optimizer = self._init_optimizer()

        scheduler = self._init_scheduler(optimizer)

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
            },
        }

    def _init_optimizer(self):
        return create_optimizer_v2(
            self.parameters(),
            opt=self.hparams.optimizer,
            lr=self.hparams.learning_rate,
            weight_decay=self.hparams.weight_decay,
        )

    def _init_scheduler(self, optimizer):
        if self.hparams.scheduler == "CosineAnnealingLR":
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=self.hparams.max_epochs,
                eta_min=self.hparams.eta_min,
            )
        elif self.hparams.scheduler == "StepLR":
            scheduler = torch.optim.lr_scheduler.StepLR(
                optimizer,
                step_size=self.hparams.max_epochs // 5,
                gamma=0.95,
            )
        else:
            raise ValueError(f"Unknown scheduler: {self.hparams.scheduler}")
        return scheduler

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def predict_step(self, batch, batch_idx):
        _, labels, logits = self._forward_pass(batch, stage=None)
        preds = logits.sigmoid()
        return preds, labels

    def _shared_step(self, batch, stage):
        x, y, y_pred = self._forward_pass(batch)

        loss = self.loss_fn(y_pred, y)

        self.metrics[f"{stage}_metrics"](y_pred, y)

        self._log(stage, loss, batch_size=len(x))

        return loss

    def _forward_pass(self, batch):
        x, y = batch
        y = y.view(-1)
        y_pred = self(x)

        return x, y, y_pred

    def _log(self, stage, loss, batch_size):
        self.log(f"{stage}_loss", loss, batch_size=batch_size)
        self.log_dict(self.metrics[f"{stage}_metrics"], batch_size=batch_size)

# Train

In [ ]:
def load_logger_and_callbacks(
    fast_dev_run, metrics, overfit_batches, patience, project, val_fold
):
    if fast_dev_run or overfit_batches > 0:
        logger, callbacks = None, None
    else:
        logger, id_ = get_logger(metrics=metrics, project=project)
        callbacks = get_callbacks(
            id_=id_,
            mode=list(metrics.values())[0],
            monitor=list(metrics.keys())[0],
            patience=patience,
            val_fold=val_fold,
        )

    return logger, callbacks


def get_logger(metrics, project):
    logger = WandbLogger(project=project)
    id_ = logger.experiment.id

    for metric, summary in metrics.items():
        logger.experiment.define_metric(metric, summary=summary)

    return logger, id_


def get_callbacks(id_, mode, monitor, patience, val_fold):
    callbacks = [
        EarlyStopping(monitor=monitor, mode=mode, patience=patience),
        LearningRateMonitor(),
    ]

    return callbacks

In [ ]:
def train(
    accelerator: str = "gpu",
    batch_size: int = 256,
    devices: int = 1,
    drop_rate: float = 0.4,
    eta_min: float = 1e-6,
    fast_dev_run: bool = False,
    first_out_features: int = 2048,
    in_features: int = IN_FEATURES,
    learning_rate: float = 3e-4,
    loss: str = "CELoss",
    max_epochs: int = 200,
    model_name: str = "linear",
    num_blocks: int = 3,
    num_classes: int = 250,
    num_workers: int = 2,
    overfit_batches: int = 0,
    optimizer: str = "AdamW",
    patience: int = 20,
    precision: int = 16,
    project: str = "asl-sign-detection-kaggle",
    scheduler: str = "CosineAnnealingLR",
    swa: bool = False,
    val_fold: float = 2.0,
    weight_decay: float = 1e-6,
):
    pl.seed_everything(SEED, workers=True)

    if fast_dev_run:
        num_workers = 0

    data_module = ASLDataModule(
        batch_size=batch_size,
        data_csv_path=TRAIN_CSV_PATH,
        in_features=in_features,
        num_workers=num_workers,
        val_fold=val_fold,
    )

    module = ASLModule(
        drop_rate=drop_rate,
        eta_min=eta_min,
        first_out_features=first_out_features,
        in_features=in_features,
        learning_rate=learning_rate,
        loss=loss,
        max_epochs=max_epochs,
        model_name=model_name,
        num_blocks=num_blocks,
        num_classes=num_classes,
        optimizer=optimizer,
        scheduler=scheduler,
        weight_decay=weight_decay,
    )

    logger, callbacks = load_logger_and_callbacks(
        fast_dev_run=fast_dev_run,
        metrics={"val_loss": "min", "val_acc": "max", "val_f1": "max"},
        overfit_batches=overfit_batches,
        patience=patience,
        project=project,
        val_fold=val_fold,
    )

    trainer = pl.Trainer(
        accelerator=accelerator,
        benchmark=True,
        devices=devices,
        callbacks=callbacks,
        fast_dev_run=fast_dev_run,
        logger=logger,
        log_every_n_steps=5,
        max_epochs=max_epochs,
        num_sanity_val_steps=0,
        overfit_batches=overfit_batches,
        precision=precision,
        strategy="ddp" if devices > 1 else None,
    )

    trainer.fit(module, datamodule=data_module)

    return module

In [ ]:
module = train()

# Tensorflow Conversion

## Source: [End-to-End Pytorch Training + Submission](https://www.kaggle.com/code/mayukh18/end-to-end-pytorch-training-submission)

In [ ]:
onnx_asl_module_path = "asl_module.onnx"

input_sample = torch.rand((1, IN_FEATURES)).cuda()
module.to_onnx(
    file_path=onnx_asl_module_path,
    input_sample=input_sample,
    opset_version=12,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "input"}},
)

In [ ]:
tf_model_path = "tf_model"
onnx_asl_module = onnx.load(onnx_asl_module_path)
tf_rep = prepare(onnx_asl_module)
tf_rep.export_graph(tf_model_path)

# Final Inference Model in Tensorflow

## Source: [End-to-End Pytorch Training + Submission](https://www.kaggle.com/code/mayukh18/end-to-end-pytorch-training-submission)

In [ ]:
class ASLInferModel(tf.Module):
    def __init__(self, tf_model_path):
        super().__init__()

        self.feature_gen = FeatureGenTF()
        self.model = tf.saved_model.load(tf_model_path)
        self.feature_gen.trainable = False
        self.model.trainable = False

    @tf.function(
        input_signature=[
            tf.TensorSpec(shape=[None, 543, 3], dtype=tf.float32, name="inputs")
        ]
    )
    def call(self, input):
        output_tensors = {}
        features = self.feature_gen(tf.cast(input, dtype=tf.float32))

        output_tensors["outputs"] = self.model(**{"input": features})["output"][0, :]

        return output_tensors


mytfmodel = ASLInferModel(tf_model_path)
tf.saved_model.save(
    mytfmodel, "tf_infer_model", signatures={"serving_default": mytfmodel.call}
)

# Submission

## Source: [End-to-End Pytorch Training + Submission](https://www.kaggle.com/code/mayukh18/end-to-end-pytorch-training-submission)

In [ ]:
tf_infer_model_path = "/kaggle/working/tf_infer_model"
converter = tf.lite.TFLiteConverter.from_saved_model(tf_infer_model_path)
tflite_model = converter.convert()

tflite_model_path = "model.tflite"

# Save the model
with open(tflite_model_path, "wb") as f:
    f.write(tflite_model)

In [ ]:
def load_relevant_data_subset(pq_path):
    data_columns = ["x", "y", "z"]
    data = pd.read_parquet(pq_path, columns=data_columns)
    n_frames = int(len(data) / ROWS_PER_FRAME)
    data = data.values.reshape(n_frames, ROWS_PER_FRAME, len(data_columns))
    return data.astype(np.float32)

In [ ]:
pq_path = "/kaggle/input/asl-signs/train_landmark_files/53618/1001379621.parquet"

import tflite_runtime.interpreter as tflite

interpreter = tflite.Interpreter(tflite_model_path)
interpreter.allocate_tensors()

found_signatures = list(interpreter.get_signature_list().keys())

# if REQUIRED_SIGNATURE not in found_signatures:
#     raise KernelEvalException('Required input signature not found.')

prediction_fn = interpreter.get_signature_runner("serving_default")
output = prediction_fn(inputs=load_relevant_data_subset(pq_path))
sign = np.argmax(output["outputs"])

print(sign, output["outputs"].shape)

In [ ]:
!zip submission.zip $tflite_model_path